# Extension: Hourly Balance Fetch — Jan-Jun 2023

Extends the hourly balance store back 6 months to give the walk-forward validation a 24-month window instead of 18 months.

**Cost: ~$7.50  |  Restart-safe: yes**

## E1. Configuration

In [1]:
from claude_refactor_core_utils import *
from datetime import date, timedelta, datetime
import pandas as pd
import numpy as np
import os

ALCHEMY_API_KEY      = "8J0Ou3_XsuWVbOiZ0xpEvioGx7k0wab9"
ETHERSCAN_API_KEY    = "GH6FVBANEIR1HS6PEN36XMYQRY2IDEAAXR"

# Extension window — 6 months before existing data
EXTENSION_START      = date(2023, 1, 1)
EXTENSION_END        = date(2023, 6, 30)

# Existing files
HOURLY_BALANCE_CSV   = '/home/colin/projects/blockchain_alpha/data/balance_store_hourly_top200.csv'
HOURLY_BLOCK_MAP_CSV = '/home/colin/projects/blockchain_alpha/data/hourly_block_map.csv'
EXISTING_BLOCK_MAP_CSV = '/home/colin/projects/blockchain_alpha/data/warm_start_block_df.csv'
DAILY_SKILL_CSV      = '/home/colin/projects/blockchain_alpha/data/random_10k_skill_results.csv'

n_hours = (EXTENSION_END - EXTENSION_START).days * 24
skill_df = pd.read_csv(DAILY_SKILL_CSV)
skill_df['address'] = skill_df['address'].str.lower().str.strip()
top200 = (
    skill_df[skill_df['horizon'] == 1]
    .sort_values('return_diff', ascending=False)
    .head(200)['address'].tolist()
)
# Also include confirmed skilled wallet
target = '0x1b4b91bb747176451f0f7f73b636babafcf31cc4'
if target not in top200:
    top200.append(target)

n_wallets = len(top200)
cost = n_wallets * n_hours * 19 / 1_000_000 * 0.45

print(f"Extension window: {EXTENSION_START} to {EXTENSION_END}")
print(f"Wallets: {n_wallets}")
print(f"Hours:   {n_hours:,}")
print(f"Pairs:   {n_wallets * n_hours:,}")
print(f"Est. cost: ${cost:.2f}")


Extension window: 2023-01-01 to 2023-06-30
Wallets: 201
Hours:   4,320
Pairs:   868,320
Est. cost: $7.42


## E2. Extend hourly block map

In [2]:
# Extend hourly block map to cover Jan-Jun 2023
# Uses interpolation from daily block map — no Etherscan calls needed

print("Extending hourly block map to cover extension window...")

daily_block_df = pd.read_csv(EXISTING_BLOCK_MAP_CSV)
daily_block_df['timestamp'] = pd.to_datetime(daily_block_df['timestamp'])
daily_block_df['block_number'] = pd.to_numeric(
    daily_block_df['block_number'], errors='coerce'
)
daily_block_df = daily_block_df.dropna(subset=['block_number']).sort_values('timestamp')

# Build full hourly index covering both extension and existing windows
full_start = EXTENSION_START
full_end   = date(2024, 12, 31)  # existing data end

hourly_index = pd.date_range(start=full_start, end=full_end, freq='h')

daily_indexed = (
    daily_block_df
    .set_index('timestamp')['block_number']
    .reindex(
        daily_block_df.set_index('timestamp').index.union(hourly_index)
    )
    .sort_index()
    .interpolate(method='index')
    .reindex(hourly_index)
    .round()
    .astype(int)
)

# Save updated block map
block_map_full = {ts: int(bn) for ts, bn in daily_indexed.items()}
block_map_df   = pd.DataFrame({
    'timestamp':    list(block_map_full.keys()),
    'block_number': list(block_map_full.values())
})
block_map_df.to_csv(HOURLY_BLOCK_MAP_CSV, index=False)

# Filter to extension window only for fetch
ext_hours  = [ts for ts in block_map_full.keys()
              if EXTENSION_START <= ts.date() <= EXTENSION_END]
block_map  = {ts: block_map_full[ts] for ts in ext_hours}

print(f"Full hourly block map: {len(block_map_full):,} hours")
print(f"Extension window hours: {len(block_map):,}")
print(f"Range: {min(block_map.keys())} to {max(block_map.keys())}")

# Sanity check
diffs = pd.Series(list(block_map.values())).diff().dropna()
print(f"Mean blocks/hour: {diffs.mean():.0f} (expected ~300)")
print(f"Monotonic: {(diffs > 0).all()}")


Extending hourly block map to cover extension window...
Full hourly block map: 17,521 hours
Extension window hours: 4,344
Range: 2023-01-01 00:00:00 to 2023-06-30 23:00:00
Mean blocks/hour: 296 (expected ~300)
Monotonic: True


## E3. Fetch Jan-Jun 2023 balances (restart-safe)

In [3]:
# ============================================================
# Extension fetch — Jan-Jun 2023, restart-safe
# Appends to existing balance store
# ============================================================

def load_already_fetched_hourly(path):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path, usecols=['address','date'])
    df['date']    = pd.to_datetime(df['date'], format='mixed').dt.floor('h')
    df['address'] = df['address'].str.lower().str.strip()
    return set(zip(df['address'], df['date']))

print("Loading already-fetched pairs...")
already_fetched = load_already_fetched_hourly(HOURLY_BALANCE_CSV)
ext_hours_sorted = sorted(block_map.keys())
total_pairs = len(top200) * len(ext_hours_sorted)
already_in_ext = sum(
    1 for addr in top200 for h in ext_hours_sorted
    if (addr.lower().strip(), pd.Timestamp(h).floor('h')) in already_fetched
)
print(f"Already fetched in extension window: {already_in_ext:,} / {total_pairs:,}")
print()

# Build reverse lookup for date fixing
block_to_hour = {v: k for k, v in block_map.items()}

fetch_count = 0
for i, hour in enumerate(ext_hours_sorted):
    hour_norm = pd.Timestamp(hour).floor('h')
    block_num = block_map.get(hour_norm)
    if block_num is None:
        continue

    sched = pd.DataFrame({'address': top200, 'date': hour_norm})
    sched['address'] = sched['address'].str.lower().str.strip()
    mask = sched.apply(
        lambda r: (r['address'], r['date']) not in already_fetched, axis=1
    )
    sched_new = sched[mask].reset_index(drop=True)

    if not sched_new.empty:
        hour_block_df = pd.DataFrame([{
            'timestamp': hour_norm, 'block_number': block_num
        }])

        bal = fetch_balances_for_schedule(
            schedule=sched_new,
            block_df=hour_block_df,
            alchemy_api_key=ALCHEMY_API_KEY,
            batch_size=100,
            max_concurrency=30,
            rps=25.0,
            progress_every=5000,
            block_date_col="timestamp",
            label=f"ext2-{str(hour_norm)[:13]}"
        )

        if not bal.empty:
            # Fix date precision using block_to_hour reverse lookup
            bal['block_number_int'] = pd.to_numeric(
                bal['block_number'], errors='coerce'
            ).astype('Int64')
            bal['date'] = bal['block_number_int'].map(block_to_hour)
            bal = bal.drop(columns=['block_number_int'])

            bal.to_csv(
                HOURLY_BALANCE_CSV, mode='a',
                header=not os.path.exists(HOURLY_BALANCE_CSV),
                index=False
            )

            bal['date_norm'] = pd.to_datetime(
                bal['date'], format='mixed'
            ).dt.floor('h')
            newly = set(zip(
                bal['address'].str.lower().str.strip(),
                bal['date_norm']
            ))
            already_fetched.update(newly)
            fetch_count += len(bal)

    if (i+1) % 24 == 0 or (i+1) == len(ext_hours_sorted):
        pct = (already_in_ext + fetch_count) / total_pairs * 100
        print(f"[{i+1}/{len(ext_hours_sorted)}] {str(hour_norm)[:13]} — "
              f"{fetch_count:,} new rows ({pct:.1f}% complete)")

print(f"\nExtension fetch complete. {fetch_count:,} new rows written.")


Loading already-fetched pairs...
Already fetched in extension window: 0 / 873,144

[ext2-2023-01-01 00] 3/3 batches (100.0%)
[ext2-2023-01-01 01] 3/3 batches (100.0%)
[ext2-2023-01-01 02] 3/3 batches (100.0%)
[ext2-2023-01-01 03] 3/3 batches (100.0%)
[ext2-2023-01-01 04] 3/3 batches (100.0%)
[ext2-2023-01-01 05] 3/3 batches (100.0%)
[ext2-2023-01-01 06] 3/3 batches (100.0%)
[ext2-2023-01-01 07] 3/3 batches (100.0%)
[ext2-2023-01-01 08] 3/3 batches (100.0%)
[ext2-2023-01-01 09] 3/3 batches (100.0%)
[ext2-2023-01-01 10] 3/3 batches (100.0%)
[ext2-2023-01-01 11] 3/3 batches (100.0%)
[ext2-2023-01-01 12] 3/3 batches (100.0%)
[ext2-2023-01-01 13] 3/3 batches (100.0%)
[ext2-2023-01-01 14] 3/3 batches (100.0%)
[ext2-2023-01-01 15] 3/3 batches (100.0%)
[ext2-2023-01-01 16] 3/3 batches (100.0%)
[ext2-2023-01-01 17] 3/3 batches (100.0%)
[ext2-2023-01-01 18] 3/3 batches (100.0%)
[ext2-2023-01-01 19] 3/3 batches (100.0%)
[ext2-2023-01-01 20] 3/3 batches (100.0%)
[ext2-2023-01-01 21] 3/3 batches (1

## E4. Verify extension

In [5]:
# Verify extension data loaded correctly
raw = pd.read_csv(HOURLY_BALANCE_CSV)
raw['date'] = pd.to_datetime(raw['date'], format='mixed').dt.floor('h')
raw['address'] = raw['address'].str.lower().str.strip()

# Deduplicate
bal = (
    raw
    .sort_values(['address','date','block_number'])
    .groupby(['address','date'], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

full_start = EXTENSION_START
full_end   = date(2024, 12, 31)
total_hours = (pd.Timestamp(full_end) - pd.Timestamp(full_start)).days * 24
expected    = n_wallets * total_hours
coverage    = len(bal) / expected * 100

print(f"Balance store after extension:")
print(f"  Total rows:       {len(bal):,}")
print(f"  Unique addresses: {bal['address'].nunique():,}")
print(f"  Date range:       {bal['date'].min()} to {bal['date'].max()}")
print(f"  Expected pairs:   {expected:,}")
print(f"  Coverage:         {coverage:.1f}%")
print()
print("Rows by period:")
bal['period'] = bal['date'].dt.to_period('M')
print(bal.groupby('period').size().to_string())


Balance store after extension:
  Total rows:       3,521,721
  Unique addresses: 201
  Date range:       2023-01-01 00:00:00 to 2024-12-31 00:00:00
  Expected pairs:   3,521,520
  Coverage:         100.0%

Rows by period:
period
2023-01    149544
2023-02    135072
2023-03    149544
2023-04    144720
2023-05    149544
2023-06    144720
2023-07    149544
2023-08    149544
2023-09    144720
2023-10    149544
2023-11    144720
2023-12    149544
2024-01    149544
2024-02    139896
2024-03    149544
2024-04    144720
2024-05    149544
2024-06    144720
2024-07    149544
2024-08    149544
2024-09    144720
2024-10    149544
2024-11    144720
2024-12    144921
Freq: M
